In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MiPrimerAnalisis")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

print(spark)
print("App:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print("Versión:", spark.version)
print("Executor mem:", spark.conf.get("spark.executor.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

App: MiPrimerAnalisis
Master: local[*]
Versión: 3.5.0
Executor mem: 1g
Shuffle partitions: 8


In [2]:
sc = spark.sparkContext
rdd = sc.parallelize(list(range(1, 21)))

print("Total de elementos:", rdd.count())   # 20
print("Primeros 5:", rdd.take(5))           # [1, 2, 3, 4, 5]
print("Suma:", rdd.sum())                   # 210
print("Máximo:", rdd.max(), "Mínimo:", rdd.min())  # 20, 1

Total de elementos: 20
Primeros 5: [1, 2, 3, 4, 5]
Suma: 210
Máximo: 20 Mínimo: 1


In [3]:
numeros = sc.parallelize(range(1, 51))

triple = numeros.map(lambda x: x * 3)
mayores = triple.filter(lambda x: x > 50)

print("Cuántos > 50:", mayores.count())     # 34
print("Primeros 10:", mayores.take(10))     # [51, 54, ..., 78]

Cuántos > 50: 34
Primeros 10: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78]


In [4]:
ventas = sc.parallelize([
    ("Laptop", 1200), ("Mouse", 25), ("Laptop", 1100),
    ("Teclado", 75), ("Mouse", 30), ("Laptop", 1300)
])

totales = ventas.reduceByKey(lambda a, b: a + b)
print("Totales:", totales.collect())        # Laptop 3600, Mouse 55, Teclado 75

conteo = ventas.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
print("Nº de ventas:", conteo.collect())    # Laptop 3, Mouse 2, Teclado 1

top = totales.max(key=lambda x: x[1])
print("Mayor total:", top)                  # ('Laptop', 3600)

Totales: [('Laptop', 3600), ('Mouse', 55), ('Teclado', 75)]
Nº de ventas: [('Laptop', 3), ('Mouse', 2), ('Teclado', 1)]
Mayor total: ('Laptop', 3600)


In [5]:
!mkdir -p DATOS
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv" -P ./DATOS
!head ./DATOS/persons.csv

--2026-11-18 17:37:34--  https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/persons.csv [following]
--2026-11-18 17:37:35--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/persons.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607 [text/plain]
Saving to: ‘./DATOS/persons.csv.5’

persons.csv.5       100%[===================>]     607  --.-KB/s    in 0s      

2026-11-18 17:37:35 (9.53 MB/s) - ‘./DATOS/persons.csv.5’ saved [607/607]

name,age,gender,salary
John Doe,30,Male,

In [6]:
import os

# Ruta local con prefijo file://
path_final = "file://" + os.path.abspath("./DATOS/persons.csv")
print("Leyendo desde:", path_final)

# Cargar (cada línea es un string)
rdd = sc.textFile(path_final)
print(rdd.take(3))

# Quitar encabezado
header = rdd.first()
sin_header = rdd.filter(lambda linea: linea != header)

# Extraer nombre y edad (split una sola vez)
def parsear(linea):
    columnas = linea.split(",")
    return (columnas[0], int(columnas[1]))

nombres_edades = sin_header.map(parsear)

# Filtrar mayores de 30
mayores = nombres_edades.filter(lambda x: x[1] > 30)

print("Personas mayores de 30:", mayores.collect())
print("Total:", mayores.count())

Leyendo desde: file:///home/jovyan/work/projects/DATOS/persons.csv
['name,age,gender,salary', 'John Doe,30,Male,50000', 'Jane Smith,25,Female,45000']
Personas mayores de 30: [('David Johnson', 35), ('Michael Wilson', 40), ('Sarah Brown', 32), ('James Martinez', 38), ('William Anderson', 33), ('Karen Hernandez', 31), ('Christopher Taylor', 37), ('Matthew Davis', 36), ('Daniel Miller', 34), ('Linda Martin', 39)]
Total: 10


In [7]:
rdd = sc.textFile("file:///ruta/datos.csv")

def parsear(linea):
    nombre, edad, ciudad = linea.split(",")   # un solo split
    return (nombre, edad, ciudad)

datos = rdd.map(parsear)   # una sola pasada, las tres columnas

# Si necesitas columnas sueltas, salen de la tupla ya parseada
nombres = datos.map(lambda x: x[0])
edades = datos.map(lambda x: x[1])
ciudades = datos.map(lambda x: x[2])

In [8]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Debugging").getOrCreate()
sc = spark.sparkContext

path = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd = sc.textFile(path)

for linea in rdd.take(100):
    print(linea)

spark.stop()

name,age,gender,salary
John Doe,30,Male,50000
Jane Smith,25,Female,45000
David Johnson,35,Male,60000
Emily Davis,28,Female,52000
Michael Wilson,40,Male,75000
Sarah Brown,32,Female,58000
Robert Lee,29,Male,51000
Lisa Garcia,27,Female,49000
James Martinez,38,Male,70000
Jennifer Rodriguez,26,Female,47000
William Anderson,33,Male,62000
Karen Hernandez,31,Female,55000
Christopher Taylor,37,Male,69000
Mary Gonzalez,24,Female,44000
Matthew Davis,36,Male,67000
Patricia White,29,Female,50000
Daniel Miller,34,Male,64000
Elizabeth Jackson,30,Female,52000
Joseph Harris,28,Male,53000
Linda Martin,39,Female,71000


In [9]:
from pyspark.sql import SparkSession
import os

# Cerrar la sesión anterior para que se aplique la nueva configuración
try:
    spark.stop()
except:
    pass

spark = (SparkSession.builder
    .appName("AnalisisStocks")
    .config("spark.executor.memory", "512m")
    .getOrCreate())

sc = spark.sparkContext

# Descargar el dataset
!mkdir -p DATOS
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt" -P ./DATOS
!head ./DATOS/stocks.txt

# Ruta absoluta con file://
path_relativo = "./DATOS/stocks.txt"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto

# Cargar y verificar
rdd = sc.textFile(path_final)
print(rdd.take(5))

--2026-11-18 17:37:39--  https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/stocks.txt [following]
--2026-11-18 17:37:40--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/stocks.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 986 [text/plain]
Saving to: ‘./DATOS/stocks.txt.6’

stocks.txt.6        100%[===================>]     986  --.-KB/s    in 0s      

2026-11-18 17:37:40 (9.24 MB/s) - ‘./DATOS/stocks.txt.6’ saved [986/986]

id,name,category,quantity,price
1,iPhone,Elec

In [10]:
def parsear_linea(linea):
    try:
        p = linea.split(",")
        # (id, nombre, categoría, cantidad, precio)
        return (int(p[0]), p[1].strip(), p[2].strip(), int(p[3]), float(p[4]))
    except:
        return None

rdd_parseado = rdd.map(parsear_linea).filter(lambda x: x is not None)

print("Líneas totales:", rdd.count())          # 31
print("Líneas válidas:", rdd_parseado.count()) # 30 (el encabezado se descarta solo)
print(rdd_parseado.take(3))

Líneas totales: 31
Líneas válidas: 30
[(1, 'iPhone', 'Electronics', 10, 899.99), (2, 'Macbook', 'Electronics', 5, 1299.99), (3, 'iPad', 'Electronics', 15, 499.99)]


In [11]:
categoria_precio = rdd_parseado.map(lambda x: (x[2], x[4]))
categoria_precio.cache()

precio_max = categoria_precio.reduceByKey(lambda a, b: max(a, b))
precio_min = categoria_precio.reduceByKey(lambda a, b: min(a, b))

precio_promedio = categoria_precio.aggregateByKey(
    (0.0, 0),
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
).mapValues(lambda x: x[0] / x[1])

print("Precio máximo por categoría:")
for cat, v in sorted(precio_max.collect()):
    print(f"{cat}: ${v:.2f}")

print("\nPrecio mínimo por categoría:")
for cat, v in sorted(precio_min.collect()):
    print(f"{cat}: ${v:.2f}")

print("\nPrecio promedio por categoría:")
for cat, v in sorted(precio_promedio.collect()):
    print(f"{cat}: ${v:.2f}")

Precio máximo por categoría:
Accessories: $29.99
Clothing: $199.99
Electronics: $1299.99
Food: $5.99
Furniture: $249.99
Sports: $49.99

Precio mínimo por categoría:
Accessories: $24.99
Clothing: $14.99
Electronics: $129.99
Food: $0.25
Furniture: $79.99
Sports: $19.99

Precio promedio por categoría:
Accessories: $27.49
Clothing: $99.28
Electronics: $586.66
Food: $2.30
Furniture: $141.99
Sports: $34.99


In [12]:
producto_valor = rdd_parseado.map(lambda x: (x[1], x[3] * x[4]))

valor_por_producto = producto_valor.reduceByKey(lambda a, b: a + b)
top_productos = valor_por_producto.sortBy(lambda x: x[1], ascending=False)

print("Top 10 productos por valor de inventario:")
for nombre, valor in top_productos.take(10):
    print(f"{nombre}: ${valor:,.2f}")

Top 10 productos por valor de inventario:
iPhone: $8,999.90
iPad: $7,499.85
LG TV: $6,999.90
Macbook: $6,499.95
Samsung TV: $6,399.92
Camera: $5,999.90
Beats Headphones: $3,999.80
Sneakers: $3,199.60
Leather Jacket: $2,999.85
Nike Shoes: $2,999.70


In [13]:
stats_por_categoria = categoria_precio.aggregateByKey(
    (float('-inf'), float('inf')),                       # (max, min)
    lambda acc, precio: (max(acc[0], precio), min(acc[1], precio)),
    lambda acc1, acc2: (max(acc1[0], acc2[0]), min(acc1[1], acc2[1]))
)

volatilidad = stats_por_categoria.mapValues(
    lambda x: ((x[0] - x[1]) / x[1]) * 100
)

mas_volatiles = volatilidad.sortBy(lambda x: x[1], ascending=False).take(5)
mas_estables = volatilidad.sortBy(lambda x: x[1], ascending=True).take(5)

print("Categorías más volátiles:")
for cat, vol in mas_volatiles:
    print(f"{cat}: {vol:.2f}%")

print("\nCategorías más estables:")
for cat, vol in mas_estables:
    print(f"{cat}: {vol:.2f}%")

Categorías más volátiles:
Food: 2296.00%
Clothing: 1234.16%
Electronics: 900.07%
Furniture: 212.53%
Sports: 150.08%

Categorías más estables:
Accessories: 20.01%
Sports: 150.08%
Furniture: 212.53%
Electronics: 900.07%
Clothing: 1234.16%


In [14]:
import os, shutil

os.makedirs("./RESULTADOS", exist_ok=True)

ruta_valor = os.path.abspath("./RESULTADOS/valor_por_producto")
ruta_stats = os.path.abspath("./RESULTADOS/stats_por_categoria")

# Borrar si ya existen (Spark no sobrescribe)
for r in (ruta_valor, ruta_stats):
    if os.path.exists(r):
        shutil.rmtree(r)

valor_por_producto.saveAsTextFile("file://" + ruta_valor)
stats_por_categoria.saveAsTextFile("file://" + ruta_stats)

# Métricas para el resumen
total_registros = rdd_parseado.count()
categorias_unicas = rdd_parseado.map(lambda x: x[2]).distinct().count()
cat_mayor_precio = precio_promedio.sortBy(lambda x: x[1], ascending=False).first()
prod_mayor_valor = valor_por_producto.sortBy(lambda x: x[1], ascending=False).first()
cat_mas_volatil = volatilidad.sortBy(lambda x: x[1], ascending=False).first()

print("\n" + "=" * 60)
print("RESUMEN EJECUTIVO - ANÁLISIS DE INVENTARIO")
print("=" * 60)
print(f"\nTotal de registros procesados: {total_registros:,}")
print(f"Número de categorías únicas: {categorias_unicas}")
print(f"\nCategoría con mayor precio promedio: {cat_mayor_precio[0]} (${cat_mayor_precio[1]:.2f})")
print(f"Producto con mayor valor de inventario: {prod_mayor_valor[0]} (${prod_mayor_valor[1]:,.2f})")
print(f"Categoría más volátil: {cat_mas_volatil[0]} ({cat_mas_volatil[1]:.2f}%)")

spark.stop()
print("\n Análisis completado. Sesión cerrada.")


RESUMEN EJECUTIVO - ANÁLISIS DE INVENTARIO

Total de registros procesados: 30
Número de categorías únicas: 6

Categoría con mayor precio promedio: Electronics ($586.66)
Producto con mayor valor de inventario: iPhone ($8,999.90)
Categoría más volátil: Food (2296.00%)

 Análisis completado. Sesión cerrada.


In [15]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Reflexion").getOrCreate()
sc = spark.sparkContext

rdd = sc.textFile("file://" + os.path.abspath("./DATOS/stocks.txt"))
print("Particiones:", rdd.getNumPartitions())

Particiones: 2


## Preguntas de reflexión

### 1. Eficiencia

**¿Por qué es importante hacer `split()` una sola vez por línea?**
Cada `split()` recorre la línea completa y crea una lista nueva. Repetirlo por columna repite trabajo ya hecho, y si además se usan varios `map` separados, Spark recorre todo el archivo una vez por cada `map`.

**¿Cuánto tiempo ahorraría con 1 millón de líneas?**
No hay una cifra fija porque depende de la máquina, pero el trabajo baja a la tercera parte con 3 columnas: de 3 millones de splits y 3 recorridos a 1 millón de splits y 1 recorrido. En mi prueba con `persons.csv` los tiempos fueron: *(escribe aquí tus tiempos)*. Con un archivo tan pequeño la diferencia casi no se nota, pero con millones de líneas sí pesa.

### 2. Particionamiento

**¿Cuántas particiones tiene mi RDD?**
*(escribe aquí el número que te dé `rdd.getNumPartitions()`)*. Con un archivo pequeño, `textFile` crea 2 por defecto. Con archivos grandes crea más, según su tamaño.

**¿Cómo afecta al rendimiento?**
Cada partición se procesa en una tarea y las tareas corren en paralelo. Si hay pocas particiones, quedan cores sin trabajo. Si hay demasiadas para pocos datos, el costo de coordinarlas supera el beneficio.

**¿Qué pasaría con una sola partición?**
Todo el trabajo se haría en una única tarea, sin paralelismo. Además, esa tarea tendría que manejar todos los datos en memoria.

### 3. Lazy evaluation

**¿En qué momento se ejecutaron realmente las transformaciones?**
Cuando se llamó a la primera acción que dependía de ellas. En mi proyecto, `textFile` se ejecutó con el `take(5)` del Paso 1, `map` y `filter` con `rdd_parseado.count()` del Paso 2, y `reduceByKey` con el `collect()` del Paso 3. Antes de eso solo se construía el plan.

**¿Qué ventajas tiene este modelo?**
Spark ve el plan completo antes de ejecutarlo. Así puede combinar pasos en una sola pasada, evitar trabajo innecesario y, si falla un nodo, recalcular solo la parte perdida.

### 4. Memoria

**¿Qué pasaría con `collect()` en un RDD de 10 GB?**
`collect()` trae todos los datos a la memoria del driver, que normalmente tiene 1 o 2 GB. Daría `OutOfMemoryError`, o un error por superar `spark.driver.maxResultSize` (1 GB por defecto), y el programa se caería.

**¿Cómo evitaría este problema?**
Usando `take(n)` para ver una muestra, agregando los datos (`reduceByKey`, `count`) antes de traer un resultado pequeño, o guardando el resultado con `saveAsTextFile(### 5. Optimización

**¿Qué cambiaría en mi código si procesara 100 veces más datos?**

- **Evitar `collect()`:** los resultados finales (como el top 10) se obtienen con `take(n)`, y los resultados grandes se guardan con `saveAsTextFile()` en lugar de traerlos al driver.
- **Parsear una sola vez:** mantener la función `parsear_linea`, para que cada línea se divida una sola vez.
- **Filtrar temprano:** descartar las líneas inválidas al inicio, para no arrastrarlas por el resto del proceso.
- **Usar `reduceByKey` y no `groupByKey`:** `reduceByKey` combina los valores dentro de cada partición antes de moverlos por la red, así que el shuffle es mucho más ligero.
- **Más recursos:** subir `spark.executor.memory` y `spark.driver.memory` y trabajar en un cluster con varios nodos.
- **Leer desde HDFS** en lugar de un archivo local, para que los datos estén distribuidos y se lean en paralelo.
- **Pasar a DataFrames** con un formato como Parquet, porque Spark los optimiza mejor que los RDDs.

**¿Usaría más particiones?**

Sí. Con 100 veces más datos, pocas particiones dejarían cores sin trabajo y cada tarea sería demasiado grande para la memoria. Una regla práctica es tener de 2 a 4 particiones por core disponible. Se ajusta al leer (`sc.textFile(ruta, minPartitions=...)`), con `repartition(n)` o pasando `numPartitions` a `reduceByKey`. En DataFrames, el equivalente es `spark.sql.shuffle.partitions`, que por defecto vale 200.

**¿Cachearía algún RDD?**

Sí, el que se reutiliza en varias acciones. En mi proyecto es `categoria_precio`, que se usa en el Paso 3 (máximo, mínimo y promedio) y en el Paso 5 (volatilidad). Sin `cache()`, Spark vuelve a leer y parsear el archivo en cada acción. Con datos muy grandes conviene `persist(StorageLevel.MEMORY_AND_DISK)`, para que lo que no quepa en RAM se guarde en disco en vez de recalcularse. No se debe cachear un RDD que se usa una sola vez, porque ocuparía memoria sin ningún beneficio.ón de las tres partes que te di antes)*

### 6. HDFS vs local

**¿Qué ventajas tiene HDFS para este análisis?**
HDFS reparte y replica los datos en varios nodos: si uno falla, Spark lee otra réplica. Los datos se leen en paralelo desde varias máquinas, y Spark puede ejecutar cada tarea en el nodo que ya tiene el bloque, sin mover datos por la red. Un archivo local solo existe en una máquina y no escala más allá de ella. Con mi archivo de 31 líneas no hay diferencia práctica; la ventaja aparece con datos grandes.